# CMP611 Progress-2 (Colab Final, Auto-Backup)

Bu notebook runtime kapanmalarına karşı Drive yedeklemeli tasarlandı.

- `human_ocr_ensembl` yok
- Ek dataset olarak sadece **`human_nontata_promoters`** var (promoter task ailesi)
- Sonuçlar her kritik adımda Drive'a kopyalanır

In [ ]:
# 0) Mount Drive + backup paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import shutil, os

TS = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_ROOT = Path('/content/drive/MyDrive/cmp611_backups')
RUN_BACKUP = DRIVE_ROOT / f'run_{TS}'
RUN_BACKUP.mkdir(parents=True, exist_ok=True)

print('Backup root:', RUN_BACKUP)

In [ ]:
# 1) Upload project.zip
from google.colab import files
files.upload()

In [ ]:
# 2) Unzip project
import zipfile, shutil, os

ZIP = '/content/project.zip'
PROJECT_DIR = '/content/project'

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(ZIP, 'r') as zf:
    zf.extractall('/content')

print('Project exists:', os.path.exists(PROJECT_DIR))
print('Top-level:', sorted(os.listdir(PROJECT_DIR))[:30])

In [ ]:
# 3) Install deps (stable)
%cd /content/project
!pip -q install -r requirements.txt
!pip -q uninstall -y peft || true
!pip -q install genomic-benchmarks==0.0.9

In [ ]:
# 4) Environment check
%cd /content/project
!python scripts/check_environment.py

In [ ]:
# 5) Utility: backup helper
from pathlib import Path
import shutil

RUN_BACKUP = Path(str(RUN_BACKUP))

def backup_path(src, dst_name=None):
    src = Path(src)
    if not src.exists():
        print('[WARN] not found:', src)
        return
    dst = RUN_BACKUP / (dst_name or src.name)
    if dst.exists():
        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print('[OK] backed up:', src, '->', dst)

In [ ]:
# 6) Download GUE + immediate backup
%cd /content/project
!python scripts/download_gue.py --out-dir data/raw
backup_path('/content/project/data/raw/GUE', 'GUE_raw')

In [ ]:
# 7) Build low-data splits for GUE tasks
%cd /content/project

RATIOS = '0.01,0.10,1.0'
SEEDS = '13'

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_300_all \
  --output-root data/low_data/prog2/prom_300_all \
  --ratios $RATIOS \
  --seeds $SEEDS

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_core_all \
  --output-root data/low_data/prog2/prom_core_all \
  --ratios $RATIOS \
  --seeds $SEEDS

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/splice/reconstructed \
  --output-root data/low_data/prog2/splice_reconstructed \
  --ratios $RATIOS \
  --seeds $SEEDS

backup_path('/content/project/data/low_data/prog2', 'low_data_prog2')

In [ ]:
# 8) Download + convert external promoter dataset: human_nontata_promoters
%cd /content/project

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from genomic_benchmarks.loc2seq import download_dataset

dest = Path('/content/gb_data')
dest.mkdir(parents=True, exist_ok=True)

_ = download_dataset('human_nontata_promoters', version=0, dest_path=dest, force_download=False)

# robust root resolve
candidates = [dest / 'human_nontata_promoters']
if _ is not None:
    candidates.insert(0, Path(_))
root = None
for c in candidates:
    if c.exists():
        root = c
        break
if root is None:
    raise FileNotFoundError('Could not locate human_nontata_promoters after download.')

print('Dataset root:', root)

def load_split(split_name):
    split_dir = root / split_name
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    if not class_dirs:
        raise RuntimeError(f'No class folders found in {split_dir}')
    label_map = {d.name: i for i, d in enumerate(class_dirs)}

    rows = []
    for cls_dir in class_dirs:
        label_id = label_map[cls_dir.name]
        for fp in cls_dir.glob('*.txt'):
            seq = fp.read_text().strip().upper()
            if seq:
                rows.append({'sequence': seq, 'label': label_id})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'Empty dataframe for split={split_name}')
    return df, label_map

train_df, label_map = load_split('train')
test_df, _ = load_split('test')

train_df, dev_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=13,
    stratify=train_df['label']
)

out = Path('/content/project/data/raw/extra/human_nontata_promoters')
out.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out / 'train.csv', index=False)
dev_df.to_csv(out / 'dev.csv', index=False)
test_df.to_csv(out / 'test.csv', index=False)

print('Saved to:', out)
print('Label map:', label_map)
print('Sizes train/dev/test:', len(train_df), len(dev_df), len(test_df))
backup_path(str(out), 'human_nontata_promoters_csv')

In [ ]:
# 9) Build low-data splits for external promoter dataset
%cd /content/project
RATIOS = '0.01,0.10,1.0'
SEEDS = '13'

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/extra/human_nontata_promoters \
  --output-root data/low_data/prog2/human_nontata_promoters \
  --ratios $RATIOS \
  --seeds $SEEDS

backup_path('/content/project/data/low_data/prog2/human_nontata_promoters', 'low_data_human_nontata_promoters')

In [ ]:
# 10) Smoke run (quick validation)
%cd /content/project
!python scripts/train_dnabert2.py \
  --model_name_or_path zhihan1996/DNABERT-2-117M \
  --data_path data/low_data/prog2/prom_core_all/r1_seed13 \
  --run_name smoke_prom_core_all_r1_seed13 \
  --output_dir outputs/smoke_test \
  --seed 13 \
  --model_max_length 70 \
  --max_steps 60 \
  --per_device_train_batch_size 8 \
  --per_device_eval_batch_size 16 \
  --gradient_accumulation_steps 1 \
  --learning_rate 3e-5 \
  --warmup_steps 5 \
  --weight_decay 0.01 \
  --evaluation_strategy steps \
  --save_strategy steps \
  --eval_steps 60 \
  --save_steps 60 \
  --logging_steps 20 \
  --load_best_model_at_end True \
  --metric_for_best_model eval_f1_macro \
  --greater_is_better True \
  --save_total_limit 1 \
  --report_to none \
  --fp16 True \
  --use_lora False

backup_path('/content/project/outputs/smoke_test', 'outputs_smoke_test')

## Main runs
Varsayılan: 4 koşu grubu (promoter, core_promoter, splice, promoter_external_nontata), her biri r1/r10/r100 seed13.

Sadece promoter odaklı gitmek istersen `TASKS` satırında filtreyi aç.

In [ ]:
# 11) Main runs
%cd /content/project
python - << 'PY'
import subprocess, os

MODEL = 'zhihan1996/DNABERT-2-117M'
BASE_OUT = 'outputs/prog2/main_runs'

TASKS = [
    ('promoter',                 'data/low_data/prog2/prom_300_all',             300, 600),
    ('core_promoter',            'data/low_data/prog2/prom_core_all',              70, 600),
    ('splice',                   'data/low_data/prog2/splice_reconstructed',      400, 600),
    ('promoter_external_nontata','data/low_data/prog2/human_nontata_promoters',   251, 600),
]

# TASKS = [t for t in TASKS if t[0] in ['promoter', 'promoter_external_nontata']]

ratios = ['r1_seed13', 'r10_seed13', 'r100_seed13']

for task_name, root, max_len, max_steps in TASKS:
    for r in ratios:
        data_path = f'{root}/{r}'
        run_name = f'{task_name}_{r}'
        out_dir  = f'{BASE_OUT}/{task_name}/{r}'
        os.makedirs(out_dir, exist_ok=True)

        cmd = [
            'python', 'scripts/train_dnabert2.py',
            '--model_name_or_path', MODEL,
            '--data_path', data_path,
            '--run_name', run_name,
            '--output_dir', out_dir,
            '--seed', '13',
            '--model_max_length', str(max_len),
            '--max_steps', str(max_steps),
            '--per_device_train_batch_size', '8',
            '--per_device_eval_batch_size', '16',
            '--gradient_accumulation_steps', '1',
            '--learning_rate', '3e-5',
            '--warmup_steps', '50',
            '--weight_decay', '0.01',
            '--evaluation_strategy', 'steps',
            '--save_strategy', 'steps',
            '--eval_steps', '200',
            '--save_steps', '200',
            '--logging_steps', '50',
            '--load_best_model_at_end', 'True',
            '--metric_for_best_model', 'eval_f1_macro',
            '--greater_is_better', 'True',
            '--save_total_limit', '1',
            '--report_to', 'none',
            '--fp16', 'True',
            '--use_lora', 'False'
        ]

        print('\nRUN:', ' '.join(cmd))
        subprocess.run(cmd, check=True)

print('\nAll main runs finished.')
PY

In [ ]:
# 12) Aggregate results (unknown bug fixed in local script)
%cd /content/project
!python scripts/aggregate_results.py \
  --results-root outputs/prog2/main_runs \
  --out-csv outputs/prog2_summary_all_runs.csv \
  --out-mean-csv outputs/prog2_summary_mean_std.csv

!python - << 'PY'
import pandas as pd
from pathlib import Path

p = Path('outputs/prog2_summary_all_runs.csv')
print('exists:', p.exists())
if p.exists():
    df = pd.read_csv(p)
    print(df[['run_name','method','ratio_percent','seed','eval_f1_macro','eval_auroc']].head(12))
    print('rows:', len(df))
PY

In [ ]:
# 13) Backup outputs to Drive + create zip
%cd /content/project

backup_path('/content/project/outputs/prog2', 'outputs_prog2')
backup_path('/content/project/outputs/prog2_summary_all_runs.csv', 'prog2_summary_all_runs.csv')
backup_path('/content/project/outputs/prog2_summary_mean_std.csv', 'prog2_summary_mean_std.csv')

!zip -r /content/prog2_outputs_final.zip outputs/prog2 outputs/prog2_summary_all_runs.csv outputs/prog2_summary_mean_std.csv
backup_path('/content/prog2_outputs_final.zip', 'prog2_outputs_final.zip')

print('Done. Drive backup folder:', RUN_BACKUP)